# Jab temporal model — self-contained Colab

This notebook needs only the JSON file in Google Drive. It does not clone GitHub or require project files.

Select **Runtime → Change runtime type → T4 GPU**, then run every cell in order.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install 'scikit-learn>=1.4,<2' 'onnx>=1.16' 'onnxruntime>=1.17' 'onnxscript>=0.1'

import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))

## Connect Drive files

The configured input is exactly `/content/drive/MyDrive/martial-art-ai/jab-synthetic-bootstrap.json`.

In [ ]:
from pathlib import Path
import json

DRIVE_ROOT = Path('/content/drive/MyDrive/martial-art-ai')
SYNTHETIC_JSON = DRIVE_ROOT / 'jab-synthetic-bootstrap.json'
REAL_DATA_DIR = DRIVE_ROOT / 'data' / 'jab'
DATASET_PATH = DRIVE_ROOT / 'datasets' / 'jab_temporal_dataset.npz'
MODEL_DIR = DRIVE_ROOT / 'models' / 'jab-v1'

REAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
DATASET_PATH.parent.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

if not SYNTHETIC_JSON.exists():
    raise FileNotFoundError(f'Missing: {SYNTHETIC_JSON}')
print('Synthetic input:', SYNTHETIC_JSON)
print('Optional real JSON directory:', REAL_DATA_DIR)
print('Model output:', MODEL_DIR)

## Load and validate sessions

Optional real Data Lab exports placed in `MyDrive/martial-art-ai/data/jab` are included automatically.

In [ ]:
def sessions_from(path):
    document = json.loads(path.read_text(encoding='utf-8'))
    sessions = document.get('sessions') if isinstance(document, dict) else None
    return sessions if isinstance(sessions, list) else [document]

sessions = sessions_from(SYNTHETIC_JSON)
for path in sorted(REAL_DATA_DIR.glob('*.json')):
    if path.resolve() != SYNTHETIC_JSON.resolve():
        sessions.extend(sessions_from(path))

origins = {}
for session in sessions:
    origin = session.get('provenance', {}).get('origin', 'real')
    origins[origin] = origins.get(origin, 0) + 1
    status = session.get('manual_annotation', {}).get('status')
    if status not in {'human_verified', 'synthetic_verified'}:
        raise ValueError(f'Unverified session: {session.get("session_id")}')

print('Sessions:', len(sessions))
print('Origins:', origins)
print('Frames:', sum(len(s.get('frames', [])) for s in sessions))

## Build normalized sequence windows

In [ ]:
import numpy as np

LABEL_NAMES = ['__PAD__', '__UNKNOWN__', '__TRACKING_LOST__', 'GUARD', 'EXTENSION', 'FULL_EXTENSION', 'RETRACTION', 'RECOVERY']
LABEL_TO_ID = {name: index for index, name in enumerate(LABEL_NAMES)}
SEQUENCE_LENGTH = 90
STRIDE = 15

def restore_pose(frame):
    values = frame.get('wp') or frame.get('op') or frame.get('p') or []
    pose = np.zeros((33, 4), dtype=np.float32)
    for i, point in enumerate(values[:33]):
        for channel, value in enumerate(point[:4]):
            pose[i, channel] = float(value) / 10000.0
    return pose

def normalize_pose(pose):
    result = pose.copy()
    root = (result[23, :3] + result[24, :3]) / 2
    shoulders = np.linalg.norm(result[11, :3] - result[12, :3])
    torso = np.linalg.norm((result[11, :3] + result[12, :3]) / 2 - root)
    scale = max(float(shoulders), float(torso), 1e-4)
    result[:, :3] = (result[:, :3] - root) / scale
    result[:, 3] = np.clip(result[:, 3], 0, 1)
    return result

def labels_for(session):
    frames = session.get('frames', [])
    labels = ['__PAD__'] * len(frames)
    for segment in session.get('manual_annotation', {}).get('segments', []):
        start, end = int(segment['start_frame']), int(segment['end_frame'])
        state = str(segment['state']).strip().upper()
        if start < 0 or end >= len(frames) or end < start:
            raise ValueError(f'Invalid segment in {session.get("session_id")}')
        for i in range(start, end + 1):
            if labels[i] != '__PAD__': raise ValueError('Overlapping segments')
            labels[i] = state if state in LABEL_TO_ID else '__UNKNOWN__'
    if any(label == '__PAD__' for label in labels):
        raise ValueError(f'Labels do not cover every frame: {session.get("session_id")}')
    return labels

features, targets, masks, groups, window_origins = [], [], [], [], []
for session_number, session in enumerate(sessions):
    frames = session.get('frames', [])
    if not frames: continue
    x_all = np.asarray([normalize_pose(restore_pose(frame)) for frame in frames], dtype=np.float32)
    y_all = np.asarray([LABEL_TO_ID[label] for label in labels_for(session)], dtype=np.int64)
    starts = list(range(0, max(1, len(frames) - SEQUENCE_LENGTH + 1), STRIDE))
    last = max(0, len(frames) - SEQUENCE_LENGTH)
    if last not in starts: starts.append(last)
    for start in starts:
        end = min(len(frames), start + SEQUENCE_LENGTH)
        valid = end - start
        x = np.zeros((SEQUENCE_LENGTH, 33, 4), dtype=np.float32)
        y = np.zeros(SEQUENCE_LENGTH, dtype=np.int64)
        mask = np.zeros(SEQUENCE_LENGTH, dtype=bool)
        x[:valid], y[:valid], mask[:valid] = x_all[start:end], y_all[start:end], True
        features.append(x); targets.append(y); masks.append(mask)
        groups.append(str(session.get('session_id') or f'session-{session_number}'))
        window_origins.append(session.get('provenance', {}).get('origin', 'real'))

features = np.stack(features)
targets = np.stack(targets)
masks = np.stack(masks)
groups = np.asarray(groups)
window_origins = np.asarray(window_origins)
np.savez_compressed(DATASET_PATH, features=features, labels=targets, mask=masks, groups=groups, origins=window_origins, label_names=np.asarray(LABEL_NAMES))
print('Features:', features.shape)
print('Windows by origin:', dict(zip(*np.unique(window_origins, return_counts=True))))
print('Saved:', DATASET_PATH)

## Define and train the ST-GCN

If at least four real sessions exist, validation and test are real-only. With the current synthetic-only file, scores are only a pipeline check.

In [ ]:
import random
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, f1_score
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
POSE_EDGES = [(0,1),(1,2),(2,3),(3,7),(0,4),(4,5),(5,6),(6,8),(9,10),(11,12),(11,13),(13,15),(15,17),(15,19),(15,21),(17,19),(12,14),(14,16),(16,18),(16,20),(16,22),(18,20),(11,23),(12,24),(23,24),(23,25),(25,27),(27,29),(29,31),(27,31),(24,26),(26,28),(28,30),(30,32),(28,32)]

def adjacency():
    matrix = np.eye(33, dtype=np.float32)
    for a, b in POSE_EDGES: matrix[a,b] = matrix[b,a] = 1
    degree = matrix.sum(1)
    inv = np.diag(np.power(np.maximum(degree, 1), -0.5))
    return torch.tensor(inv @ matrix @ inv, dtype=torch.float32)

class GraphBlock(nn.Module):
    def __init__(self, incoming, outgoing):
        super().__init__(); self.proj=nn.Linear(incoming,outgoing); self.norm=nn.LayerNorm(outgoing); self.drop=nn.Dropout(.2)
    def forward(self, x, adj): return self.drop(torch.nn.functional.gelu(self.norm(self.proj(torch.einsum('vw,btwc->btvc',adj,x)))))

class TemporalModel(nn.Module):
    def __init__(self, classes):
        super().__init__(); self.register_buffer('adj',adjacency()); self.g1=GraphBlock(4,48); self.g2=GraphBlock(48,96)
        self.temporal=nn.Sequential(nn.Conv1d(96,96,5,padding=2),nn.BatchNorm1d(96),nn.GELU(),nn.Dropout(.2),nn.Conv1d(96,96,5,padding=4,dilation=2),nn.BatchNorm1d(96),nn.GELU(),nn.Dropout(.2)); self.head=nn.Linear(96,classes)
    def forward(self,x):
        spatial=self.g2(self.g1(x,self.adj),self.adj); visibility=x[...,3].clamp(0,1).unsqueeze(-1)
        pooled=(spatial*visibility).sum(2)/visibility.sum(2).clamp_min(1); return self.head(self.temporal(pooled.transpose(1,2)).transpose(1,2))

def grouped_split(indexes):
    first=GroupShuffleSplit(n_splits=1,test_size=.30,random_state=SEED); train,hold=next(first.split(indexes,groups=groups[indexes]))
    second=GroupShuffleSplit(n_splits=1,test_size=.50,random_state=SEED+1); val_local,test_local=next(second.split(hold,groups=groups[indexes][hold]))
    return indexes[train],indexes[hold[val_local]],indexes[hold[test_local]]

all_ids=np.arange(len(groups)); real_ids=all_ids[window_origins!='synthetic']; synthetic_ids=all_ids[window_origins=='synthetic']
if len(np.unique(groups[real_ids])) >= 4:
    real_train,val_ids,test_ids=grouped_split(real_ids); train_ids=np.concatenate([real_train,synthetic_ids]); print('Real-only validation/test enabled')
else:
    train_ids,val_ids,test_ids=grouped_split(all_ids); print('WARNING: synthetic-only evaluation; pipeline check only')

def loader(ids,shuffle):
    ds=TensorDataset(torch.from_numpy(features[ids]).float(),torch.from_numpy(targets[ids]).long(),torch.from_numpy(masks[ids]).bool())
    return DataLoader(ds,batch_size=32,shuffle=shuffle)

train_loader,val_loader,test_loader=loader(train_ids,True),loader(val_ids,False),loader(test_ids,False)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); model=TemporalModel(len(LABEL_NAMES)).to(device)
values=targets[train_ids][masks[train_ids]]; counts=np.bincount(values,minlength=len(LABEL_NAMES)).astype(float); weights=counts.sum()/np.maximum(counts,1); weights/=max(weights.mean(),1e-6); weights[0]=0
criterion=nn.CrossEntropyLoss(weight=torch.tensor(weights,dtype=torch.float32,device=device),ignore_index=0); optimizer=torch.optim.AdamW(model.parameters(),lr=3e-4,weight_decay=1e-4)

@torch.no_grad()
def evaluate(data_loader):
    model.eval(); predictions=[]; truth=[]
    for x,y,mask in data_loader:
        x,y,mask=x.to(device),y.to(device),mask.to(device); pred=model(x).argmax(-1); valid=mask & y.ne(0)
        predictions.extend(pred[valid].cpu().tolist()); truth.extend(y[valid].cpu().tolist())
    return f1_score(truth,predictions,average='macro',zero_division=0),truth,predictions

best=-1; patience=10; stale=0; checkpoint=MODEL_DIR/'best_temporal_phase.pt'
for epoch in range(1,61):
    model.train(); losses=[]
    for x,y,mask in train_loader:
        x,y,mask=x.to(device),y.to(device),mask.to(device); optimizer.zero_grad(set_to_none=True); logits=model(x); loss=criterion(logits.reshape(-1,len(LABEL_NAMES)),y.masked_fill(~mask,0).reshape(-1)); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),1); optimizer.step(); losses.append(loss.item())
    score,_,_=evaluate(val_loader); print(f'epoch={epoch:03d} loss={np.mean(losses):.4f} validation_f1={score:.4f}')
    if score>best: best=score; stale=0; torch.save(model.state_dict(),checkpoint)
    else:
        stale+=1
        if stale>=patience: print('Early stopping'); break

model.load_state_dict(torch.load(checkpoint,map_location=device)); test_f1,y_true,y_pred=evaluate(test_loader)
report={'test_macro_f1':float(test_f1),'validation_best_macro_f1':float(best),'evaluation_origin':'real' if len(np.unique(groups[real_ids]))>=4 else 'synthetic_pipeline_check','classification_report':classification_report(y_true,y_pred,labels=list(range(1,len(LABEL_NAMES))),target_names=LABEL_NAMES[1:],zero_division=0,output_dict=True)}
(MODEL_DIR/'test_report.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
print('Test macro F1:',test_f1); print('Saved report:',MODEL_DIR/'test_report.json')

## Export ONNX and runtime metadata

In [ ]:
model=model.cpu().eval(); dummy=torch.zeros(1,SEQUENCE_LENGTH,33,4,dtype=torch.float32); onnx_path=MODEL_DIR/'temporal_phase_classifier.onnx'
torch.onnx.export(model,dummy,onnx_path,input_names=['landmarks'],output_names=['state_logits'],dynamic_axes={'landmarks':{0:'batch'},'state_logits':{0:'batch'}},opset_version=18,dynamo=False)
metadata={'schema_version':'1.0','model_version':'jab-temporal-v1','technique_id':'jab','sequence_length':SEQUENCE_LENGTH,'landmark_layout':'mediapipe-pose-33','input_channels':['x','y','z','visibility'],'state_names':LABEL_NAMES[1:],'output':'per_frame_state_logits','test_macro_f1':float(test_f1),'evaluation_origin':report['evaluation_origin']}
metadata_path=MODEL_DIR/'temporal_phase_classifier.metadata.json'; metadata_path.write_text(json.dumps(metadata,indent=2),encoding='utf-8')
print('ONNX:',onnx_path,onnx_path.stat().st_size/1024,'KB'); print('Metadata:',metadata_path)

In [ ]:
import onnxruntime as ort
session=ort.InferenceSession(str(onnx_path),providers=['CPUExecutionProvider']); output=session.run(None,{'landmarks':dummy.numpy()})[0]
print('ONNX validation output:',output.shape)
print('All artifacts:')
for path in sorted(MODEL_DIR.glob('*')): print(path.name,round(path.stat().st_size/1024,1),'KB')

## Download the trained runtime files

They also remain saved in `MyDrive/martial-art-ai/models/jab-v1`.

In [ ]:
from google.colab import files
files.download(str(onnx_path))
files.download(str(metadata_path))

**Important:** With only synthetic sessions, the result verifies the training/export pipeline but is not proof of accuracy on people. Add human-verified Data Lab exports to the real-data directory before production evaluation.